# SemLayoutDiff Server — Kaggle Deployment

Runs the `backend_new` FastAPI server on Kaggle GPU and exposes it via ngrok.

**Endpoints (same as original backend):**
- `POST  /pipeline/normalize-run`
- `GET   /pipeline/normalize-run/{job_id}/status`
- `GET   /pipeline/normalize-run/{job_id}/result`

In [ ]:
# List Kaggle input files
import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

## Step 1 — HuggingFace Login
Required to download model checkpoints.

In [ ]:
from huggingface_hub import login
login("hf_XXXXXXXXXXXXXXXXXXXXXXXX")

## Step 2 — Clone Server Repository

In [ ]:
import os

REPO_DIR = "/kaggle/working/semlayoutdiff-server"

if not os.path.exists(REPO_DIR):
    !git clone https://github.com/MinhmpcNguyen/semlayoutdiff-server.git {REPO_DIR}
else:
    print("Repo already cloned — pulling latest...")
    !cd {REPO_DIR} && git pull

!ls {REPO_DIR}

## Step 3 — Install Dependencies

Kaggle already provides: `torch`, `torchvision`, `numpy`, `opencv-python`, `Pillow`, `scipy`, `scikit-image`

We only need to add the server libs + the `semlayoutdiff` package itself.

In [ ]:
# Server framework + ML utilities
!pip install -q \
    "fastapi>=0.111.0" \
    "uvicorn[standard]>=0.29.0" \
    "httpx>=0.27.0" \
    "pydantic>=2.7.0" \
    "omegaconf>=2.3.0" \
    "openai>=1.30.0" \
    "cryptography>=42.0.0" \
    pyngrok \
    pytorch-lightning \
    einops \
    tqdm

print("✅ Dependencies installed")


In [ ]:
# Install the semlayoutdiff package from the cloned repo
!pip install -q -e {REPO_DIR}
print("✅ semlayoutdiff package installed")

## Step 4 — Download Model Checkpoints

In [ ]:
import os

CKPT_DIR = "/kaggle/working/checkpoints"
os.makedirs(CKPT_DIR, exist_ok=True)

SLDN_TAR  = f"{CKPT_DIR}/sldn_checkpoints.tar.gz"
SLDN_DIR  = f"{CKPT_DIR}/sldn_checkpoints"
APM_CKPT  = f"{CKPT_DIR}/apm_checkpoint.ckpt"

HF_BASE = "https://huggingface.co/datasets/3dlg-hcvc/semlayoutdiff/resolve/main/weights"

# SLDN checkpoints
if not os.path.exists(SLDN_DIR):
    print("Downloading SLDN checkpoints...")
    !wget -q "{HF_BASE}/sldn_checkpoints.tar.gz" -O {SLDN_TAR}
    !tar -xf {SLDN_TAR} -C {CKPT_DIR}/
    print("✅ SLDN checkpoints downloaded")
else:
    print("✅ SLDN checkpoints already present")

# APM checkpoint
if not os.path.exists(APM_CKPT):
    print("Downloading APM checkpoint...")
    !wget -q "{HF_BASE}/apm_checkpoint.ckpt" -O {APM_CKPT}
    print("✅ APM checkpoint downloaded")
else:
    print("✅ APM checkpoint already present")

print("\nCheckpoint files:")
!ls -lh {CKPT_DIR}/
!ls {SLDN_DIR}/

## Step 5 — Set Environment Variables

In [ ]:
import os

os.environ["SLDN_MODEL_DIR"]      = "/kaggle/working/checkpoints/sldn_checkpoints"
os.environ["APM_CHECKPOINT"]      = "/kaggle/working/checkpoints/apm_checkpoint.ckpt"
os.environ["METADATA_DIR"]        = f"{REPO_DIR}/preprocess/metadata"
os.environ["JOB_STORAGE_DIR"]     = "/kaggle/working/server_jobs"
os.environ["NUM_OPTIONS"]         = "3"
os.environ["SLDN_CONDITION_TYPE"] = "arch"
os.environ["SLDN_DEBUG_DIR"]      = "/kaggle/working/server_jobs/debug_latest"

# Azure OpenAI — same env vars as the original backend.
# The API key is stored encrypted in private_key.enc (already in the repo).
# Add ONE Kaggle secret named OPENAI_PUBLIC_KEY (the Fernet key).
# Kaggle Secrets: Notebook Settings → Add-ons → Secrets → + Add secret
#
# Fixed values (no secret needed — safe to hardcode):
os.environ["AZURE_OPENAI_ENDPOINT"]          = "https://auto-furniture-dev-resource.services.ai.azure.com/api/projects/auto-furniture-dev/openai/v1/responses"
os.environ["AZURE_OPENAI_API_VERSION"]        = "2024-12-01-preview"
os.environ["AZURE_OPENAI_CHAT_DEPLOYMENT"]    = "gpt-5.4-mini-furniture"
os.environ["AZURE_OPENAI_PRIMARY_DEPLOYMENT"] = "gpt-5.4-mini-furniture"
# Path to encrypted key file (checked into the repo)
os.environ["OPENAI_PRIVATE_KEY_FILE"] = f"{REPO_DIR}/private_key.enc"

# Load the Fernet decryption key from Kaggle secrets
try:
    from kaggle_secrets import UserSecretsClient
    _pk = UserSecretsClient().get_secret("OPENAI_PUBLIC_KEY")
    if _pk:
        os.environ["OPENAI_PUBLIC_KEY"] = _pk
        print("✅ OPENAI_PUBLIC_KEY loaded — LLM completion enabled")
    else:
        print("⚠️  OPENAI_PUBLIC_KEY secret is empty — LLM step disabled (rule-based fallback)")
except Exception as _e:
    print(f"⚠️  Could not load OPENAI_PUBLIC_KEY: {_e} — rule-based fallback")

print("
Environment variables set:")
for k in ["SLDN_MODEL_DIR", "APM_CHECKPOINT", "METADATA_DIR",
          "JOB_STORAGE_DIR", "NUM_OPTIONS", "SLDN_CONDITION_TYPE",
          "AZURE_OPENAI_ENDPOINT", "AZURE_OPENAI_CHAT_DEPLOYMENT"]:
    print(f"  {k}={os.environ.get(k, "(not set)")}")
print(f"  OPENAI_PUBLIC_KEY={("<set>" if os.environ.get("OPENAI_PUBLIC_KEY") else "(not set)")}")


## Step 6 — Start FastAPI Server

Runs uvicorn as a subprocess. Model loading takes ~2 minutes (SLDN + APM on GPU).

In [ ]:
import subprocess
import sys
import time
import os

SERVER_PORT = 8001
LOG_FILE    = "/kaggle/working/server.log"

# Kill any existing server on that port
os.system(f"fuser -k {SERVER_PORT}/tcp 2>/dev/null || true")
time.sleep(1)

log_fh = open(LOG_FILE, "w")
server_proc = subprocess.Popen(
    [sys.executable, "-m", "uvicorn",
     "server.app:app",
     "--host", "0.0.0.0",
     "--port", str(SERVER_PORT),
     "--log-level", "info"],
    cwd=REPO_DIR,
    env=os.environ.copy(),
    stdout=log_fh,
    stderr=subprocess.STDOUT,
)
print(f"Server PID: {server_proc.pid}")
print("Waiting for server startup + model loading (~2 min)...")

In [ ]:
import time
import urllib.request

SERVER_URL = f"http://127.0.0.1:{SERVER_PORT}"
MAX_WAIT   = 300  # seconds
INTERVAL   = 10

for elapsed in range(0, MAX_WAIT, INTERVAL):
    time.sleep(INTERVAL)
    try:
        resp = urllib.request.urlopen(f"{SERVER_URL}/", timeout=3)
        import json
        health = json.loads(resp.read())
        print(f"\n✅ Server ready after {elapsed+INTERVAL}s")
        print(f"   models_loaded : {health.get('models_loaded')}")
        print(f"   catalog_types : {health.get('catalog_types')}")
        break
    except Exception:
        # Print last few log lines while waiting
        with open(LOG_FILE) as f:
            lines = f.readlines()
        last = [l.rstrip() for l in lines[-3:]]
        print(f"[{elapsed+INTERVAL}s] waiting... last log: {last[-1] if last else ''}")
else:
    print("\n❌ Server did not start in time — check logs below")

In [ ]:
# View server startup logs
!tail -40 {LOG_FILE}

## Step 7 — Expose via ngrok

Get a free auth token at https://dashboard.ngrok.com → Your Authtoken

In [ ]:
from pyngrok import ngrok, conf

# ⚠️  Replace with your own ngrok auth token from https://dashboard.ngrok.com
NGROK_AUTH_TOKEN = "3EDkxufpmSvwLEIaDy3QCrCX6Qm_5bVLzTCvfQCa5YVzVdowQ"

ngrok.set_auth_token(NGROK_AUTH_TOKEN)

# Kill existing tunnels
for tunnel in ngrok.get_tunnels():
    ngrok.disconnect(tunnel.public_url)

tunnel = ngrok.connect(SERVER_PORT, "http")
PUBLIC_URL = tunnel.public_url

print(f"\n🌐 Public URL: {PUBLIC_URL}")
print(f"\nAPI endpoints:")
print(f"  POST  {PUBLIC_URL}/pipeline/normalize-run")
print(f"  GET   {PUBLIC_URL}/pipeline/normalize-run/{{job_id}}/status")
print(f"  GET   {PUBLIC_URL}/pipeline/normalize-run/{{job_id}}/result")
print(f"\nFrontend env var:")
print(f"  AUTO_FILL_ROOM_FURNITURE_BASE_URL={PUBLIC_URL}")

## Step 8 — Test Health Check

In [ ]:
import requests
import json

resp = requests.get(f"{SERVER_URL}/")
print(f"Status: {resp.status_code}")
print(json.dumps(resp.json(), indent=2))

## Step 9 — Test Pipeline (Bedroom, ~3 min)

Sends a sample bedroom layout request and polls until the result is ready.

In [ ]:
import requests
import time
import json

BASE = SERVER_URL  # or PUBLIC_URL for external access

payload = {
    "room": {
        "name": "Bedroom",
        "polygons": [[-3, -2], [-3, 3], [3, 3], [3, -2]],
        "description": "A simple bedroom"
    },
    "walls": [],
    "openings": [],
    "source_unit": "m",
    "tenant_id": "demo",
    "user_id": "demo_user",
    "style": "modern",
    "split_largest_room": False,
    "allow_generated_accessories": False
}

# Start job
resp = requests.post(f"{BASE}/pipeline/normalize-run", json=payload)
resp.raise_for_status()
job = resp.json()
job_id = job["id"]
print(f"Job started: {job_id}")
print(f"  status URL : {job['statusUrl']}")

# Poll status (SLDN takes ~160s)
t0 = time.time()
while True:
    status_resp = requests.get(f"{BASE}/pipeline/normalize-run/{job_id}/status")
    status = status_resp.json()
    elapsed = int(time.time() - t0)
    print(f"[{elapsed}s] status={status['status']}", flush=True)
    if status["status"] in ("ready", "error"):
        break
    time.sleep(10)

# Fetch result
if status["status"] == "ready":
    result_resp = requests.get(f"{BASE}/pipeline/normalize-run/{job_id}/result")
    result = result_resp.json()
    objects = result.get("objects", [])
    options = result.get("options", [])
    print(f"\n✅ Success! {len(objects)} objects, {len(options)} options")
    if objects:
        print("\nFirst object:")
        print(json.dumps(objects[0], indent=2))
else:
    print(f"\n❌ Job failed: {status}")
    print("Check server logs:")
    !tail -30 {LOG_FILE}

## Keep Server Running

The ngrok tunnel stays alive as long as the Kaggle session is active (max ~9 hours).
Point the frontend to `AUTO_FILL_ROOM_FURNITURE_BASE_URL=<PUBLIC_URL>` printed above.

In [ ]:
print(f"🚀 Server running at: {SERVER_URL}")
print(f"🌐 Public URL:        {PUBLIC_URL}")
print()
print("Frontend env var to set:")
print(f"  AUTO_FILL_ROOM_FURNITURE_BASE_URL={PUBLIC_URL}")

## Debug Utilities

In [ ]:
# Live tail of server logs
!tail -60 /kaggle/working/server.log

In [ ]:
# Restart server if needed (re-run after changing env vars)
import os, subprocess, sys, time

try:
    server_proc.terminate()
    server_proc.wait(timeout=10)
except Exception:
    pass

os.system(f"fuser -k {SERVER_PORT}/tcp 2>/dev/null || true")
time.sleep(2)

log_fh = open(LOG_FILE, "w")
server_proc = subprocess.Popen(
    [sys.executable, "-m", "uvicorn",
     "server.app:app",
     "--host", "0.0.0.0",
     "--port", str(SERVER_PORT),
     "--log-level", "info"],
    cwd=REPO_DIR,
    env=os.environ.copy(),
    stdout=log_fh,
    stderr=subprocess.STDOUT,
)
print(f"Restarted — PID: {server_proc.pid}")

### Update Server (git pull + restart)
Chạy cell này mỗi khi có commit mới trên GitHub.

In [ ]:
import subprocess, os, signal, time

# 1. Pull latest code
result = subprocess.run(
    ["git", "-C", REPO_DIR, "pull", "origin", "main"],
    capture_output=True, text=True
)
print(result.stdout)
if result.stderr:
    print("STDERR:", result.stderr)

# 2. Kill the running server (uvicorn)
kill_result = subprocess.run(["pkill", "-f", "uvicorn"], capture_output=True)
print(f"Killed uvicorn (returncode={kill_result.returncode})")
time.sleep(2)

# 3. Restart server in background
log_path = "/kaggle/working/server.log"
server_proc = subprocess.Popen(
    ["python", "-m", "uvicorn", "server.app:app",
     "--host", "0.0.0.0", "--port", "8001"],
    cwd=REPO_DIR,
    stdout=open(log_path, "w"),
    stderr=subprocess.STDOUT,
    env={**os.environ},
)
print(f"Server restarted (pid={server_proc.pid}). Waiting 10s for startup...")
time.sleep(10)

# 4. Health check
import urllib.request
try:
    resp = urllib.request.urlopen("http://localhost:8001/")
    import json as _j
    data = _j.loads(resp.read())
    print("✓ Server healthy:", data)
except Exception as e:
    print(f"✗ Health check failed: {e} — check {log_path}")


### Full Debug Cell
Chạy cell này sau khi server đã ready. Paste toàn bộ output để gửi cho AI phân tích.

In [ ]:
# ═══════════════════════════════════════════════════════
# FULL DEBUG — chạy cell này rồi paste toàn bộ output
# ═══════════════════════════════════════════════════════
import requests, json, time, os, glob

BASE = SERVER_URL
SEP  = '─' * 60

# ── 1. Health check ──────────────────────────────────────
print(SEP)
print('1. HEALTH CHECK')
r = requests.get(f'{BASE}/')
print(f'   HTTP {r.status_code}')
print(json.dumps(r.json(), indent=2))

# ── 2. POST normalize-run ────────────────────────────────
print(SEP)
print('2. POST /pipeline/normalize-run')
payload = {
    'room': {
        'name': 'Bedroom',
        'polygons': [[-3,-2],[-3,3],[3,3],[3,-2]],
        'description': 'debug bedroom'
    },
    'walls': [], 'openings': [],
    'source_unit': 'm', 'tenant_id': 'debug',
    'user_id': 'debug_user', 'style': 'modern',
    'split_largest_room': False,
    'allow_generated_accessories': False
}
r = requests.post(f'{BASE}/pipeline/normalize-run', json=payload)
print(f'   HTTP {r.status_code}')
job = r.json()
print(json.dumps(job, indent=2))
job_id = job.get('id', '')

# ── 3. Poll status until done ────────────────────────────
print(SEP)
print('3. POLLING STATUS (every 15s, timeout 5min)')
t0 = time.time()
last_status = None
for _ in range(20):
    time.sleep(15)
    r2 = requests.get(f'{BASE}/pipeline/normalize-run/{job_id}/status')
    last_status = r2.json()
    elapsed = int(time.time() - t0)
    st = last_status.get('status')
    print(f'   [{elapsed}s] status={st} stage={last_status.get("stage")} msg={last_status.get("message")}')
    if st in ('ready', 'error'):
        break

print('Final status response:')
print(json.dumps(last_status, indent=2))

# ── 4. Job file on disk ──────────────────────────────────
print(SEP)
print('4. JOB FILE ON DISK')
job_dir = f'/kaggle/working/server_jobs/{job_id}'
for fname in ['job.json', 'result.json']:
    fpath = f'{job_dir}/{fname}'
    if os.path.exists(fpath):
        with open(fpath) as fh:
            content = json.load(fh)
        # truncate result.json to first object only
        if fname == 'result.json' and 'objects' in content:
            content['objects'] = content['objects'][:1]
            content['options'] = content.get('options', [])[:1]
        print(f'   {fpath}:')
        print(json.dumps(content, indent=2))
    else:
        print(f'   {fpath}: NOT FOUND')

# ── 5. Relevant server logs ──────────────────────────────
print(SEP)
print('5. SERVER LOGS (lines containing job_id or ERROR/WARNING)')
with open(LOG_FILE) as fh:
    for line in fh:
        line = line.rstrip()
        if job_id in line or 'ERROR' in line or 'WARNING' in line or 'Traceback' in line:
            print('  ', line)

print(SEP)
print('DEBUG COMPLETE — paste everything above this line')


### Visualize SLDN Input (Floor Plan — Arch Encoding)
Hiển thị ảnh 120×120 đưa vào SLDN với arch encoding:
- **Trắng** = background
- **Xám** = sàn (floor)
- **Đỏ** = cửa đi (door)
- **Xanh dương** = cửa sổ (window)

Nếu không thấy pixel đỏ/xanh → server chưa `git pull` commit mới nhất.

In [ ]:
import os, json
import numpy as np
import matplotlib.pyplot as plt
from matplotlib.colors import ListedColormap
from matplotlib.patches import Patch

DEBUG_DIR = os.environ.get("SLDN_DEBUG_DIR", "/kaggle/working/server_jobs/debug_latest")
fp_path   = f"{DEBUG_DIR}/floor_plan.npy"
meta_path = f"{DEBUG_DIR}/meta.json"

if not os.path.exists(fp_path):
    print(f"❌  {fp_path} chưa tồn tại.")
    print("   → Chạy 1 job thực từ frontend trước.")
else:
    img  = np.load(fp_path)   # [120,120]  values: 0=bg, 1=floor, 2=door, 3=window
    meta = json.load(open(meta_path)) if os.path.exists(meta_path) else {}
    scale_px = meta.get("scale_px", 10.0)
    center   = tuple(meta.get("center", [0, 0]))

    unique_vals = np.unique(img)
    has_arch = any(v in unique_vals for v in [2, 3])

    # Arch color map: 0=white(bg), 1=lightgray(floor), 2=red(door), 3=cornflowerblue(window)
    arch_cmap = ListedColormap(["white", "#cccccc", "#e74c3c", "#3498db"])
    arch_legend = [
        Patch(color="white",         label="0 — bg"),
        Patch(color="#cccccc",       label="1 — floor"),
        Patch(color="#e74c3c",       label="2 — door"),
        Patch(color="#3498db",       label="3 — window"),
    ]

    fig, axes = plt.subplots(1, 2, figsize=(12, 5.5))
    title_suffix = "  [ARCH encoding: floor=1 door=2 window=3]" if has_arch else "  [binary: chỉ floor=1]"

    for ax, data, xlabel, ylabel, title in [
        (axes[0], img.T,  "→ world X  (frontend horizontal)",  "↓ world Z  (frontend depth)",
         f"SLDN Input — chiều frontend{title_suffix}"),
        (axes[1], img,    "→ world Z",  "↓ world X",
         f"SLDN Input — raw (như model nhận){title_suffix}"),
    ]:
        ax.imshow(data, cmap=arch_cmap, vmin=0, vmax=3,
                  interpolation="nearest", origin="upper")
        ax.set_xlabel(xlabel)
        ax.set_ylabel(ylabel)
        ax.set_title(title, fontsize=9)
        ax.legend(handles=arch_legend, loc="lower right", fontsize=7)
        for i in range(0, 121, 10):
            ax.axhline(i - 0.5, color="steelblue", lw=0.3, alpha=0.3)
            ax.axvline(i - 0.5, color="steelblue", lw=0.3, alpha=0.3)

    fig.suptitle(f"center={center}  |  {1/scale_px:.3f} m/px  |  {scale_px:.1f} px/m",
                 fontsize=9)
    plt.tight_layout()
    plt.show()

    n_floor  = int((img == 1).sum())
    n_door   = int((img == 2).sum())
    n_window = int((img == 3).sum())
    print(f"Floor  pixels: {n_floor:5d} ({100*n_floor/120**2:.1f}%)")
    print(f"Door   pixels: {n_door:5d}  ← {'✓ arch encoding active' if n_door > 0 else '✗ không có door pixel — server chưa pull commit mới?'}")
    print(f"Window pixels: {n_window:5d}  ← {'✓' if n_window > 0 else '✗ không có window pixel'}")
    print(f"scale_px={scale_px:.2f} px/m  →  1 pixel ≈ {1/scale_px:.2f} m")
    print(f"room_center = {center}")


### Visualize SLDN Output (Semantic Layout Map)
Load SLDN trên **CPU** (tránh chiếm VRAM của server) → chạy 1 mẫu → hiển thị ảnh màu theo category.  
⚠️ Lần đầu load mất ~30–60 s.

In [ ]:
import os, sys, json
sys.path.insert(0, REPO_DIR)
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches

DEBUG_DIR = os.environ.get("SLDN_DEBUG_DIR", "/kaggle/working/server_jobs/debug_latest")
METADATA  = os.environ["METADATA_DIR"]

sem_paths = sorted(f for f in [
    f"{DEBUG_DIR}/semantic_map_{i}.npy" for i in range(10)
] if os.path.exists(f))

if not sem_paths:
    print(f"❌  Không tìm thấy semantic_map_*.npy trong {DEBUG_DIR}")
    print("   → Chạy 1 job thực từ frontend hoặc cell 'Test Pipeline' trước.")
else:
    meta     = json.load(open(f"{DEBUG_DIR}/meta.json")) if os.path.exists(f"{DEBUG_DIR}/meta.json") else {}
    scale_px = meta.get("scale_px", 10.0)

    with open(f"{METADATA}/color_palette.json") as f:
        palette = json.load(f)
    with open(f"{METADATA}/unified_idx_to_generic_label.json") as f:
        idx_to_label = json.load(f)

    idx_to_color = {
        int(k): [c / 255.0 for c in palette.get(v, [200, 200, 200])]
        for k, v in idx_to_label.items()
    }

    def sem_to_rgb(sem):
        rgb = np.ones((120, 120, 3), dtype=np.float32) * 0.93
        for idx, color in idx_to_color.items():
            rgb[sem == idx] = color
        return rgb

    n = len(sem_paths)
    # 2 rows: top = khớp frontend (transposed), bottom = raw
    fig, axes = plt.subplots(2, n + 1, figsize=(5 * n + 3, 10),
                              gridspec_kw={"width_ratios": [1] * n + [0.5]})

    all_ids = set()
    for col_i, path in enumerate(sem_paths):
        sem = np.load(path)
        rgb = sem_to_rgb(sem)
        opt = os.path.basename(path).replace("semantic_map_", "Opt ").replace(".npy", "")

        # Top row: transposed (khớp frontend)
        axes[0, col_i].imshow(rgb.transpose(1, 0, 2), interpolation="nearest", origin="upper")
        axes[0, col_i].set_title(f"{opt} — khớp frontend\n(X→, Z↓)")
        axes[0, col_i].set_xlabel("→ world X")
        axes[0, col_i].set_ylabel("↓ world Z")
        for i in range(0, 121, 10):
            axes[0, col_i].axhline(i - 0.5, color="white", lw=0.3, alpha=0.3)
            axes[0, col_i].axvline(i - 0.5, color="white", lw=0.3, alpha=0.3)

        # Bottom row: raw
        axes[1, col_i].imshow(rgb, interpolation="nearest", origin="upper")
        axes[1, col_i].set_title(f"{opt} — raw (model nhận)\n(row=X, col=Z)")
        axes[1, col_i].set_xlabel("→ world Z")
        axes[1, col_i].set_ylabel("↓ world X")
        for i in range(0, 121, 10):
            axes[1, col_i].axhline(i - 0.5, color="white", lw=0.3, alpha=0.3)
            axes[1, col_i].axvline(i - 0.5, color="white", lw=0.3, alpha=0.3)

        all_ids.update(int(v) for v in np.unique(sem))

    # Legend column
    unique_ids = sorted(i for i in all_ids if i in idx_to_color)
    patches = [
        mpatches.Patch(color=idx_to_color[i], label=f"{i:2d}: {idx_to_label.get(str(i), '?')}")
        for i in unique_ids
    ]
    for row_i in range(2):
        axes[row_i, -1].axis("off")
    axes[0, -1].legend(handles=patches, loc="center", fontsize=9,
                       title="ID : Category", title_fontsize=10)

    plt.tight_layout()
    plt.show()

    sem0 = np.load(sem_paths[0])
    print(f"\nCategory breakdown — Option 0:")
    for uid in sorted(int(v) for v in np.unique(sem0)):
        count   = int((sem0 == uid).sum())
        area_m2 = count * (1.0 / scale_px) ** 2
        print(f"  {uid:2d}: {idx_to_label.get(str(uid), '?'):28s} {count:5d} px  ≈ {area_m2:.2f} m²")